In [1]:
%run Latex_macros.ipynb

<IPython.core.display.Latex object>

# Policy gradient based methods: concepts

Recall the Policy Gradient Theorem
- formulated with single reward $\rewseq(\tau)$ for the entire trajectory

$$
\nabla_\theta J(\theta) = 
\Exp{\tau \sim \pr{\theta}} { 
\sum_{\tt=0}^{|\tau|} {
\nabla_\theta   \log \pi(\actseq_{\tau,\tt} | \stateseq_{\tau,\tt}) ) \, \,\rewseq(\tau)
}
} 
$$

- formulated with intermediate rewards

$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{\tt=0}^{T-1} \nabla_\theta \log \pi_\theta(\actseq_{\tau,\tt} | \stateseq_{\tau,t}) G_{\tau,\tt} \right]
$$

Policy based methods
update parameters $\theta$
- in the direction (gradient)
- that increases expected return 

# Advantage: Beyond Rewards

The Policy Gradient Theorem states that the gradient for the trajectory $\tau$ is the unweighted sum
- of per step gradients


But perhaps the policy should be 
- *reinforced (positively or negatively)*
- in different proportions ?


Suppose all returns are positive.
- the Policy Gradient Theorem would positively reinforce *all* actions with positive gradients
- regardless of the magnitude of the per-step reward

Or suppose we always get "easy" questions correct and only sometimes get "hard" questions correct
- the Policy Gradient Theorem would positively reinforce the already correct easy questions
- with as much force as it positively reinforces the hard question
    - for methods that involve multiple trajectories per example (e.g., GRPO)


We need some way to *relativize* 
 the return for a state
- by comparing it to a *baseline* (the anticipated return) for the state.
    - often $\statevalfun_\pi ( \stateseq_{\tt})$, the value of the state as would be computed by a Value function
- we call this relativized value the *advantage*

We will use the advantage as a mechanism to differentially weight the gradient of individual actions.





For example: consider a baseline that is the average return from the state
- above average returns have positive advantage
- below average returns have a negative advantage

Thus, in the case of all returns being positive
- we favor actions that result in "above expected" return (positive advantages)
- we disfavor actions that result "below expected" return (negative advantages)

Subtracting a baseline from the return has other advantages
- reduces the *magnitude* of the parameter update
    - smoother training
- reduces "noise"
    - the baseline value for a state is common to *all actions* from the state
    - so the advantage is relative to the signal *only* from the action itself


| Aspect                    | Effect of Subtracting Baseline                         |
|:---------------------------|:-------------------------------------------------------|
| Noise cancellation        | Removes expected (common) return, reducing fluctuations |
| Bias introduction         | None, baseline does not depend on action              |
| Zero-centering            | Advantages centered around zero, stabilizing updates  |



## Credit Assignment: The Evolution of the Advantage

"Early" Policy based methods (e.g., REINFORCE) were a direct expression of the Policy Gradient Theorem


$$
\nabla_\theta J(\theta) = 
\Exp{\tau \sim \pr{\theta}} { 
\sum_{\tt=0}^{|\tau|} {
\nabla_\theta   \log \pi(\actseq_{\tau,\tt} | \stateseq_{\tau,\tt}) ) \, \,\rewseq(\tau)
}
} 
$$

where the gradient of each step $\tt$ was equally weighted ( by the episode reward ($\rewseq(\tau)$).



The problem with this
- some steps are "good": would have led to high episode return
- some steps are "bad":  decreased the potential episode return

But, with sparse rewards-
- we don't know how to *assign credit* for individual steps

For example:

- If an LLM writes a 100-word masterpiece 
- but makes a typo on the very last character
- the final reward might be low. 

Equal weighting of gradients punishes *all* steps whereas a single step is clearly to blame.

By obfuscating good signals from bad signals, learning is slowed
- high variance

Computing an Advantage per step
- is a means of solving the *Credit Assignment* problem
- creating differential degrees of reinforcement

"Modern RL" is associated with a pivot away from Reward and toward Advantage.
- [Asynchronous Advantage Actor-Critic](https://arxiv.org/abs/1602.01783) 2016

Having a separate Advantage $\advseq_{\tau,\tt}$ for each step $\tt$
- dictates the "Force" and "Direction" of the gradient weight update

| Advantage Value | Effect on Gradient | The "Steering" Result |
| :--- | :--- | :--- |
| **Positive ($\advseq_{\tau,\tt} > 0$)** | Multiplies by a positive scalar. | **Accelerate:** This action was better than the "usual." Make it more likely. |
| **Negative ($\advseq_{\tau,\tt} < 0$)** | Flips the sign of the gradient. | **Brake:** This action fell below expectations. Make it less likely. |
| **Zero ($\advseq_{\tau,\tt} \approx 0$)** | Nullifies the gradient. | **Maintain:** You did exactly what was expected. No learning required. |

Good steps are encouraged; bad (e.g., below expectation) steps are discouraged

The Policy Based methods we will study take different approaches to *Credit Assignment* (computing Advantage).


- REINFORCE
    - equal credit
- PPO (Surgical Credit)
    - assigns credit to each step
    - by maintaining a prediction of the value of each state (a Value function)
- GRPO (Statistical Credit)
    - Uses the **Group Mean** ($\mu$) of multiple completions as the baseline. 
    - It gives the **same Advantage** to every token in a specific trajectory. 


---

## Unified Policy Gradient Formulation: Advantage Definitions

Even though different Policy Based methods compute Advantage differently
- we can uniformly implement weighted Policy Gradient updates
- by rewriting the Policy Gradient Theorem in terms of Advantage

$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{\tt=0}^{T-1} \nabla_\theta \log \pi_\theta(\actseq_{\tau,\tt} | \stateseq_{\tau,t}) \advseq_{\tau,\tt} \right]
$$

This allows us to focus on the *Advantage definition* 
$$
\advseq_{\tau,\tt} 
$$
as a way of differentiating methods.

We thus refer to the above formulation as the Unified Policy Gradient Formulation.

The policy-based methods we present will typically be variations of this "vanilla" gradient.

# Surrogate Loss

Rather than
- maximization of return

Policy methods often switch to
- minimization of a *surrogate loss*

The surrogate loss 
- supplements the (negative of) return
- with **constraints** on the derived policy and  on the solution process.

Whereas the  Policy Gradient Theorem (Return Maximization) provides
- the *theoretical* basis for an optimal policy
- using the Returns or Advantages

the Surrogate Loss provides
- the *practical* objective for finding the optimal policy
- subject to practical constraints: stability, convergence

The actual implementation of our models will
usually revert to the Minimization of Surrogate Loss formulation.

What are some constraints we typically encounter ?

We often see *limitations* on how far an updated policy can change
- prevents drastic policy shifts after a parameter update
- by constraining the new policy to be close to a "reference" policy

Other constraints try to promote stability and convergence in the solution process
- limiting the magnitude of a gradient update
- promoting low variance of the gradient updates

**Constraints**

| Reason                          | Explanation                                               |
|:---------------------------------|:-----------------------------------------------------------|
| Trust Region Constraints        | Avoid overly large policy updates that destabilize learning |
| Regularization                  | KL divergence or clipping adds penalty to maintain stability  |
| Stability & Convergence         | Ensures smooth, incremental improvement of the policy       |
| Computational Tractability      | Easier to optimize surrogate than raw return objective      |
| Exploration-Exploitation Tradeoff | Clipping controls how much policy can change per step      |

**Relationship between Policy Gradient Theorem and Surrogate Loss**

| Aspect                  | Policy Gradient Theorem                                     | Surrogate Loss                                               |
|:------------------------|:------------------------------------------------------------|:-------------------------------------------------------------|
| Role                    | Theoretical formula for expected return gradient            | Practical approximation/objective for policy update          |
| Gradient                | Directly gives unbiased gradient of $J(\theta)$             | Its gradient approximates policy gradient but includes stabilizing terms |
| Constraints             | None intrinsic; pure gradient formula                        | Includes clipping, penalties to enforce trust region and avoid large steps |
| Use in Algorithms       | Foundation for policy gradient methods                       | Used in PPO-style algorithms to compute safe gradient steps   |
| Goal                    | Maximize expected return $J(\theta)$                         | Provide stable, incremental improvement proxy                |


Here is a preview of the Surrogate Losses of the policy-based methods we will explore.

| Method     | Surrogate Return/Objective Used for Gradient                | Credit Assignment         |
|:------------|:------------------------------------------------------------|:--------------------------|
| REINFORCE  | $G_t$ (return-to-go from $t$)                          | Monte Carlo trajectory return |
| PPO        | Clipped ratio times $A_t$ (advantage)                    | Advantage-based, stable  |
| GRPO       | Relative advantages over candidate

# Policy gradient based methods: Preview

We will show several common policy-based methods.
- REINFORCE
- PPO (the "classic" method)
- GRPO (relatively new: 2024)

Details of each follow.

As a preview, we show the definition of the Advantage for each.


| Method  | Intermediate Reward Advantage $\advseq_{\tau, \tt}$      | Single Trajectory Reward Advantage $\advseq_\tau$         |
|:---------|:--------------------------------------------|:--------------------------------------------------|
| REINFORCE | $\advseq_t = G_t - b_t$                        | $\advseq = R(\tau) - b$                              |
| PPO     | $\advseq_t = \hat{A}_t$ (e.g., GAE estimator) | $\advseq = R(\tau) - b$ applied per step             |
| GRPO    | Relative advantages per candidate           | Same per-candidate advantage applied per step    |


And a comparison:

| Method | Gradient-Based | Main Objective | Key Characteristics | Stability & Sample Efficiency | Typical Application Domains |
|:--------|:----------------|:----------------|:---------------------|:------------------------------|:-----------------------------|
| **PPO** (Proximal Policy Optimization) | Yes | Maximize expected reward with clipped surrogate objective | Uses policy gradients with clipping to limit policy update magnitude, balancing exploration and exploitation | High stability; more sample efficient than vanilla policy gradients; widely used for continuous and discrete control tasks | Robotics, games, continuous control, benchmark RL tasks |
| **DPO** (Direct Preference Optimization) | Yes | Directly optimize policy based on preference data | Uses a preference-based loss to train policy without explicit reward modeling; bypasses traditional RL complexities | Improved stability by leveraging human preferences; avoids some issues of reward misspecification | Alignment of language models with human preferences, NLP-focused RL |
| **GRPO** (Group Relative Policy Optimization) | Yes | Optimize policy using group-relative advantage estimates | Does not require a separate value function; updates policy based on relative advantages within a group of candidate outputs | More memory efficient, stable; effective for large-scale policy optimization with reduced critic reliance | Training large language models, large-scale policy optimization |
| **REINFORCE** | Yes | Maximize expected cumulative reward by direct policy gradient | Uses Monte Carlo sampled returns, applies likelihood ratio trick; pure policy gradient without value function | High variance and sample inefficient; simpler but less stable than actor-critic or PPO | Fundamental policy gradient algorithm, baseline for many RL studies |


# REINFORCE (the "Original")

REINFORCE is an early method (historically)
that is very close to the Policy Gradient Theorem formulation
- uses Advantage rather than the episode Return
- the Advantage subtracts a "baseline" from the episode Return

| Method  | Intermediate Reward Advantage $A_{\tau, \tt}$      | Single Trajectory Reward Advantage $A_\tau$         |
|:---------|:--------------------------------------------|:--------------------------------------------------|
| REINFORCE | $\advseq_\tt = G_\tt - b_\tt$                        | $\advseq = R(\tau) - b$                              |
|

The purpose of subtracting a baseline is to
- reduce the magnitude of the gradients
- reduces the variance of the gradients
- and hence: the change in policy parameters $\theta$

In  single,  end-of-episode reward, the Advantage is typically computed as
$$
\advseq_{\tau,\tt} = \left( \rewseq(\tau) - b \right)
$$

where $b$ is a baseline
- computed as the Global Moving Average of rewards
- over *all steps in the trajectory of all prompts*
- processed thus far


The Baseline Update Rule: 
- Every time you finish a rollout and get a reward $G$, 
- update the baseline:
$$b_{new} = (1 - \alpha)b_{old} + \alpha G$$


The baseline calibrates the reward received for the current prompt/trajectory against all
previous prompts.

With intermediate rewards, the Advantage is typically computed as
$$
\advseq_{\tau,\tt} = \left( G_{\tau, \tt} - b_\tt \right)
$$
- where $b_\tt$ is often a proxy for the *expected* Value function $\statevalfun_\pi(\stateseq_\tt)$ of state $\stateseq_\tt$
- requires a separate Critic to approximate the Value function

It uses "Monte-Carlo sampling" method to estimate the gradient

**Note**

"Monte Carlo sampling" is a confusing way of saying: complete trajectory for the single prompt

The word "Monte Carlo" is thus used in the same sense as 
"Monte Carlo" in the [Temporal Difference methods we studied](RL_Value_based_model_free.ipynb#Monte-Carlo-(MC)-method-(not-a-TD-method))

## Pseudo code for REINFORCE

    # REINFORCE training for LLM
    for prompt in training_prompts:
        output = llm.generate(prompt)
        reward = evaluate_output(output) # Human or automated score
        logprob = llm.logprob(output, prompt)

        # Monte Carlo policy gradient update (no critic)
        baseline = compute_baseline() # Optional: running mean for variance reduction
        loss = -logprob * (reward - baseline)
        loss.backward()
        optimizer.step()


## Discussion

REINFORCE is very close to the vanilla Policy Gradient.

But it is considered high variance
- due to a source not related to the baseline


The Advantage depends on
$$G_{\tt, \tau}$$

which is computed *based on the single trajectory*  $\tau$ that was encountered.

Thus, we are observing the Environment's Transition Probabilities 
- via a *single sample* from its distribution, for each step

So our estimate of the Transition Probabilities are  high variance.

Subsequent methods will attempt to mitigate this source of variance.


    
In the *practical* implementation of REINFORCE
- we estimate $G_{\tt}$
- over a *batch* of episodes $\tau_1, \ldots, $ for the same prompt

so we have *multiple samples*  from the probability distribution for the State/Action pair $(\state, \act)$.
- hopefully leading to a lower variance approximation

The "practical" version below will be very similar to the GRPO method we subsequently introduce.

    # Practical RL Loop Logic
    for prompt in batch:
        # 1. Sample K trajectories for the SAME prompt
        trajectories = model.generate(prompt, n=8) 

        # 2. Get rewards
        rewards = reward_model(trajectories)

        # 3. Calculate Local Baseline (The "Modern" Way)
        # This cancels out prompt difficulty automatically
        baseline = rewards.mean() 

        # 4. Standardized Advantage
        advantages = (rewards - baseline) / (rewards.std() + 1e-8)

        # 5. Compute loss weighted by how much better/worse each sample was
        loss = -log_probs * advantages

### Comparison: Theoretical vs. Practical REINFORCE

| Feature | Theoretical REINFORCE | Practical (e.g., GRPO / Batch RL) |
| :--- | :--- | :--- |
| **Rollouts per Prompt** | **1** | **Multiple (K = 4 to 64)** |
| **Baseline Type** | **Global Moving Average** (Historical) | **Local Group Mean** (Immediate) |
| **Variance** | **Very High** (Requires small learning rates) | **Reduced** (Brute-forced by sampling) |
| **Update Frequency** | Every single trajectory | After a group/batch of trajectories |
| **Compute Cost** | Low per update | High per update (heavy generation) |
| **Analogy** | Checking your lifetime golf average. | Comparing 10 swings from the same spot. |

# PPO (the "Classic")


## Intutition

PPO optimizes a **clipped surrogate** loss
- which can be interpreted as variant of the standard policy gradient theorem.

The goal of the surrogate objective is to control how rapidly the policy changes from iteration to iteration of training.
- avoid large policy shifts
- variance reduction
- monotonic improvement of policy

## Surrogate Loss for PPO

The Surrogate Loss for PPO is
$$
J_{\mathrm{PPO}}(\theta) = \mathbb{E}_\tt \left[
\min{} \left(
r_\tt(\theta) \hat{A}_\tt,\;
\mathrm{clip}\left(r_\tt(\theta), 1 - \epsilon, 1 + \epsilon\right) \hat{\advseq}_\tt
\right)
\right]
$$

We will explain how this potentially intimidating equation came about.

We express the 
- relative change in policy (for a given State/Action pair)
    - of the current Actor
    - compared to the Old Actor (Actor at start of current iteration)
- via the *probability ratio*
$$
r_\tt(\theta) = \frac{\pi_\theta(\actseq_\tt |\stateseq_\tt )}{\pi_{\theta_{\rm old}}(\actseq_\tt |\stateseq_\tt)}
$$

where $\pi_{\theta_{\rm old}}$ is the *Old Actor policy*


The Old Actor policy is the policy in effect at the start of each iteration of training
- before this iteration modifies the policy parameters
- it is the policy *under which examples for the current batch* have been generated
    - the *Behavior Policy* in On-Policy/Off-Policy terminology

During each iteration of training
- episodes/trajectories are generated  using the *Old Actor* policy
    - stored in the "experience buffer" (when PPO is applied in batches vs single examples)
    - are used *multiple times* during the iteration
- the policy parameters are updated based on the surrogate loss for these trajectories
- the updated policy becomes the Old Actor policy for the next iteration

By keeping the probability ratio close to $1$, we constrain the Gradient Descent update step to a small change in policy.


How to we keep the probability ratio close to $1$ ?
- by using clipping to constrain it to the range $[1 - \epsilon, 1 + \epsilon ]$ for small $\epsilon$

this is the *clipped* part of the clipped surrogate objective.

Formally

The *surrogate loss* in PPO is:

$$
L_{\text{sur}}(\theta) = \mathbb{E}_\tt \left[ r_\tt(\theta) \hat{\advseq}_\tt \right] 
= \mathbb{E}_\tt \left[ \frac{\pi_\theta(\actseq_\tt|\stateseq_\tt)}{\pi_{\theta_{\rm old}}(\actseq_\tt|\stateseq_\tt)} \hat{\advseq}_\tt \right]
$$

where

- $ r_\tt(\theta)$  is the probability ratio,
- $\hat{\advseq}_t$ is the advantage estimate at step $t$.



We modify the Surrogate Loss to limit the effect of the Probability Ratio.

The *clipped surrogate loss* is

$$
L_{\text{clip}}(\theta) = \mathbb{E}_\tt \left[ \min{} \left( r_\tt(\theta) \hat{\advseq}_\tt, \; \mathrm{clip}\left(r_\tt(\theta), 1 - \epsilon, 1 + \epsilon \right) \hat{\advseq}_\tt \right) \right]
$$

By minimizing the clipped surrogate loss
- we maximize $J(\theta)$ (our goal in Gradient Ascent)
- in a controlled manner



## Iterations vs Epochs

You may have noticed our introduction of an unfamiliar word: *iteration*

In Supervised Learning
- there is a fixed set of examples for the entirety of training
    - targets are fixed
- an *epoch* processes all the examples

In Reinforcement Learning
- training is based on episodes/trajectories
    - episodes/trajectories: the response to an initial state (e.g., "prompt" in LLM terms)
    - generated by an Actor policy
- the Actor policy gets updated during training (within each epoch)
    - thus, even though the "prompts" are fixed
    - *new episodes* are generated by an updated Actor
- so we have *iterations*
    - **episodes fixed** within an iteration
        - create at the start of the iteration
        - using the updated Actor
    - epochs process the *latest* set of examples
        - we repeat the epoch multiple times
 

Why do we repeat epochs multiple times within an iteration ?
 
- sample efficiency

Creating a long episode is expensive
- we want to learn as much as possible before discarding it


## Relation to the Unified Gradient Formulation

The (unclipped) Surrogate Loss does not resemble the  Unified Gradient Formulation.
- absence of the term

$$
\log \pi_\theta(\actseq_{\tau,\tt} | \stateseq_{\tau,t}) 
$$

that multiplies the Advantage.

Moreover,
 there is an extra term
$$
r_\tt(\theta) = \frac{\pi_\theta(\actseq_\tt |\stateseq_\tt )}{\pi_{\theta_{\rm old}}(\actseq_\tt |\stateseq_\tt)}
$$

in the form of the Probability Ratio.


But, these terms appear inside the Expectation of the Gradient and, by using the Likelihood Ratio trick
$$
\begin{array} \\
\nabla_\theta r_\tt(\theta) & = & r_\tt (\theta) \nabla_\theta \log r_\tt (\theta) & \text{Likelihood Ratio trick} \\
& = & r_\tt (\theta) \nabla_\theta \left( 
    \log \pi_\theta(\actseq_\tt|\stateseq_\tt) - \log \pi_{\rm old} (\actseq_\tt|\stateseq_\tt) 
\right)  & \text{log of ratio converted to difference in logs} \\
& = & r_\tt (\theta) \nabla_\theta \log \pi_\theta(\actseq_\tt|\stateseq_\tt) & \text{since } \log \pi_{\rm old} \text{ is not a function of } \theta \\
\end{array}
$$

Thus, taking the gradient of the Probability Ratio
- yields the usual
$$
\log \pi_\theta(\actseq_{\tau,\tt} | \stateseq_{\tau,t}) 
$$

term.



We still have an extra $r_\tt (\theta)$ multiplicative term (relative to the Unified Gradient Formulation).

So we don't obtain an *identical* expression
- but we note that, with clipping, $r_\tt (\theta) \approx 1$

## Advantage for PPO

To complete the presentation in terms of the Unified Gradient formulation, we define the Advantage.

| Reward Setting         | Advantage Estimate                                  |
|:-----------------------|:-----------------------------------------------------|
| Intermediate Rewards  | $\hat{\advseq}_t = G_t -  \statevalfun_\pi(\stateseq_\tt)$ or GAE                 |
| Single Total Reward   | $\hat{\advseq}_t = R(\tau) - b$   |



We need to do a "deeper dive" on PPO in order to truly understand (and appreciate)
the Advantage definition.

In short
- PPO uses a Critic NN to estimate a Value function.e.g., the
$$
\statevalfun_\pi(\stateseq_\tt)
$$
term in the table above

OR

a Local Surprise
$$
\delta_{t}
$$
in the GAE description below.


**Using Return-to-go and value estimate:**

$
\hat{\advseq}_t = G_t - \statevalfun_\pi(\stateseq_\tt)
$

where 
- $G_t$ is the *actual* return-to-go starting from time step $t$
    - in the trajectory that was encountered
- $\statevalfun_\pi(\stateseq_\tt)$ is the value of $G_t$ that was anticipated by the Value function

That is $\hat{\advseq}_t$ 
- is a measure of the error between actual fact and previously assumed value.



We can define Local Surprise $\delta_t$
- as an *attribution of the error*

to each step in the trajectory

$
\hat{\advseq}_t = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}
$

where

$
\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)
$

with

- $\gamma$ as the discount factor,
- $\lambda \in [0, 1]$ controlling the bias-variance trade-off.

**Aside**

- When $\lambda = 0$, GAE reduces to the **1-step Temporal Difference (TD) advantage**:

- When $\lambda = 1$, GAE reduces to the **Monte Carlo advantage**:

GAE smoothly interpolates between 
- high-bias/low-variance and low-bias/high-variance advantage estimates

making it very effective in practice.

## Pseudo code for PPO

There are many subtleties to the code, which we discuss below:

    # --- INITIALIZATION ---
    # pi_theta: Training Actor (weights: theta)
    # v_phi:    Training Critic (weights: phi)
    # pi_old:   Rollout Snapshot (weights: theta_old)
    # pi_ref:   Safety Anchor (weights: frozen)

    for iteration = 1, 2, ... do:

        # -------------------------------------------------------------------------
        # 1. ROLLOUT PHASE (Sampling)
        # -------------------------------------------------------------------------
        # ACTION BY: [ACTOR (pi_theta)]
        # We generate trajectories using the live model weights
        actions, states, rewards = Rollout(pi_theta)

        # -------------------------------------------------------------------------
        # 2. PRE-COMPUTATION (Frozen Stats for Optimization)
        # -------------------------------------------------------------------------
        with torch.no_grad():
            # CALCULATED BY: [OLD ACTOR (pi_old)]
            # Anchor the original action probabilities for the ratio calculation
            logp_old = pi_old(actions | states)

            # CALCULATED BY: [CRITIC (v_phi)]
            # Get baseline "expected rewards" to center the training signal
            values_old = v_phi(states)

            # CALCULATED BY: [REFERENCE (pi_ref)]
            # Establish the "standard" behavior probability to check for drift
            logp_ref = pi_ref(actions | states)

            # [ADVANTAGE CALCULATION]
            # Combines Rollout rewards (Env) and Critic predictions (Baseline)
            # Goal: Reduce Environment Variance
            A_hat = Compute_GAE(rewards, values_old)
            Returns = A_hat + values_old

        # -------------------------------------------------------------------------
        # 3. OPTIMIZATION PHASE (Learning)
        # -------------------------------------------------------------------------
        for epoch = 1 to K do:

            # CALCULATED BY: [ACTOR (pi_theta)]
            # Get logprobs of the SAME actions under CURRENTLY UPDATING weights
            logp_curr = pi_theta(actions | states)

            # CALCULATED BY: [CRITIC (v_phi)]
            # Get current value predictions to compare against actual Returns
            v_curr = v_phi(states)

            # --- LOSS CALCULATION ---

            # [ACTOR LOSS] - Uses pi_theta (logp_curr) & pi_old (logp_old) & Critic (A_hat)
            ratio = exp(logp_curr - logp_old)
            L_clip = -min(ratio * A_hat, clip(ratio, 1-eps, 1+eps) * A_hat)

            # [KL PENALTY] - Uses pi_theta (logp_curr) & pi_ref (logp_ref)
            L_kl = beta * (logp_curr - logp_ref)

            # [CRITIC LOSS] - Uses v_phi (v_curr) & actual trajectory Returns
            L_value = c1 * MSE(v_curr, Returns)

            # [UPDATE]
            # Calculate gradients for theta and phi
            Total_Loss = L_clip + L_kl + L_value
            Update theta, phi via SGD

        # -------------------------------------------------------------------------
        # 4. SYNCHRONIZATION
        # -------------------------------------------------------------------------
        # Update pi_old for the next iteration's ratio anchor
        pi_old.load_state_dict(pi_theta.state_dict())

### Key observations about the code

**Iterations vs epochs**

- Examples for the iteration are created at the start of the iteration
- The model that creates the examples
    - changes at the end of the iteration
    - **not** within the iteration
- the Actor and Critic are updated
    - **within each epoch** of **each iteration**
- the Actor **does not** create new episodes using the updated policy
    - it uses the episodes created at start of iteration
    - evaluates *new probabilities* on the states/actions of the start-of-iteration episode

**Multiple models are computing values: be aware of which does what**        

There is **one set of examples** per iteration
- ROLLOUT phase
    - creates examples that are **fixed** for this iteration
    - created by the Actor from end of previous iteration
        - this Actor be called *Old Actor* subsequent to the ROLLOUT phase
    - *Old Actor* is the **Behavior Policy** (in On-line/Offline terminology)
      
- PRE-COMPUTATION phase
    - we now call the model that create the examples the OLD ACTOR
    - the Advantage is computed with respect to the OLD ACTOR
        - using rewards and states from the episodes (examples) created by OLD ACTOR
        - NOT the rewards, states generated by the current (updated) ACTOR model
        - Using the Critic that existed at end of prior *epoch*
            - **not** prior iteration
            - to compute state values

- OPTIMIZATION phase
    - the episodes remain the ones generated by the OLD ACTOR
        - **not** using updated policy of the ACTOR
        - so the states/actions/rewards are the same
        - but the *probabilities* are updated
            - used in the ratio

**Loss function**

Contains multiple terms
- `L_clip` and `L_kl` are part of Actor Loss
    - updates Actor parameters $\theta$
- `L-value` are part of Critic Loss (MSE)
    - updates Critic parameters $\phi$
    
So both Actor and Critic
- are updated each *epoch*

The Old Actor
- changes at end of *iteration*
    - to the Actor model as updated at end of the iteration

## Trust Region Policy Optimization (TRPO) Theory (Brief)

Constraining the probability ratio
$$
r_t(\theta) = \frac{\pi_\theta(\actseq_\tt |\stateseq_\tt )}{\pi_{\theta_{\rm old}}(\actseq_\tt |\stateseq_\tt)}
$$
to the small range $$[1 - \epsilon, 1 + \epsilon]$$

- reduces the magnitude of Gradient  updates
- thus reducing the variance of Gradient updates
- and results in
monotonic improvement of the optimization objective.

Constraining the "Actor" versus "Old" policy is based on
 *Trust Region Policy Optimizaton (TRPO)*.

We state TRPO briefly.

TRPO 
- limits the change between two policies
- "Old" and "Actor" (current)

under the assumption that we "trust" the "Old"
- it is the Behavior Policy that generated the examples

It *constrains* the optimization
- to maximize a surrogate objective 

$$
\max{}_\theta \quad \mathbb{E}_{s,a \sim \pi_{\theta_{\text{old}}}} \left[
    \frac{\pi_\theta(a|s)}{\pi_{\theta_{\text{old}}}(a|s)} A^{\pi_{\theta_{\text{old}}}}(s,a)
\right]
$$
- subject to a constraint 
    - on how much the new policy can deviate from the old policy:
    $$
\mathbb{E}_{s \sim \pi_{\theta_{\text{old}}}} \left[
    D_{\mathrm{KL}}\big( \pi_{\theta_{\text{old}}}(\cdot|s) \,\|\, \pi_\theta(\cdot|s) \big)
\right] \leq \delta
$$

where:

- $\pi_{\theta_{\text{old}}}$ is the old  policy,
- $\pi_\theta$ is the new policy parameterized by $\theta$,
- $A^{\pi_{\theta_{\text{old}}}}(s,a)$ is the advantage function with respect to the old policy,
- $D_{\mathrm{KL}}(\cdot \| \cdot)$ is the Kullback-Leibler (KL) divergence measuring the difference between the old and new policies,
- $\delta$ is a small positive constant controlling the maximum allowed policy update.

With regards to the "trust region":

TRPO
- uses the KL divergence term as a constraint

PPO
- uses the *clipping* of the ratio term as a constraint


PPO can be interpreted as a practical approximation of TRPO 
- that replaces the hard KL constraint
- with a clipped probability ratio 



**Technical note**

The first argument in KL term of  TRPO
- is $\pi_\text{old}$
- **not** $\pi_\theta$

The KL function is *asymmetric*
- the expectation is taken over $\pi_\text{old}$ not $\pi_\theta$

This is because "Old" is the behavior policy -- it is our "empirical distribution" of training
- any action introduced by the updated policy
- which had $0$ probability in "Old"
- *has no direct effect* on the KL term
    - since it has $0$ weight
    - only an indirect effect by shifting probability from elsewhere

# GRPO (the "new kid")

GRPO is a Policy Gradient based methods that uses *preferences* rather than Rewards.

It will be presented in a separate module with other Preference based methods.

In [2]:
print("Done")

Done
